### Install dependencies

In [16]:
!pip install pypdf google-genai PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 67.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
import os
import json
import re
import time
from pypdf import PdfReader, PdfWriter
import google.genai as genai
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [19]:
# Run BEFORE the splitting cell to understand PDF structure
SOURCE_PDF = "/content/drive/MyDrive/CVs.pdf"
reader = PdfReader(SOURCE_PDF)
for i in range(min(15, len(reader.pages))):
    text = (reader.pages[i].extract_text() or "")[:200].strip()
    print(f"\nPage {i+1} ========================")
    print(text)


Page 1 ========================
Professional Qualification
Name MUHAMMAD SALMAN
QAMAR
Father's /GuardianQAMAR UZ ZAMAN
Date/Place of
Birth:
10-Nov-1988 /
PAKISTANI
Father’s
Occupation:
Govt Servant
Spouse Name:Aamna Bibi Spouse
Occu

Page 2 ========================
Civil Experience
Research
Publications
Name of Qualification: Passing Year: Institution Name:
Name of Post Organization Location Duration of Employment
Program CoordinatorQurtuba University, D.I.Khan

Page 3 ========================
References
Improvement of Traveling
Salesman Problem Solution
using Hybrid Algorithm based
on Best-Worst Ant System
and Particle Swarm
Optimization
Muhammad Salman QamarShanshan Tu ∗ ,
Farman Ali ∗ ,

Page 4 ========================


Page 5 ========================
Name Aamina Akbar Father's /Guardian Brig Dr. M. Akbar
Date/Place of Birth:18-Nov-1987 / US Father’s Occupation: Pak Army
Spouse Name: Abdullah Usman SiddiquiSpouse Occupation:Govt Servant
Current Sal

Page 6 ========================

In [20]:
import re
def is_new_cv_start(page_text: str) -> bool:
    """
    A new CV starts on pages like page 0, 4, 8 —
    they begin with 'Name [CAPS NAME]' pattern.
    Pages 1,2,5,6,9 are continuation pages (start with section headers).
    Pages 4,8 are blank separators.
    """
    text = page_text.strip()

    #Blank page = not a new CV start
    if len(text) < 10:
        return False

    #New CV starts with "Name" followed by capital letters at the very top
    return bool(re.match(r"^Name\s+[A-Z]", text))

In [21]:
def sanitize_filename(name: str) -> str:
    return re.sub(r'[^\w\s-]', '', name).strip().replace(' ', '_')

def is_new_cv_start(page_text: str) -> bool:
    #every first page has this line
    return "Candidate for the Post" in page_text

def extract_name_from_text(page_text: str) -> str:
    #PyMuPDF gives us: "Name\nMUHAMMAD SALMAN\nQAMAR\nFather's..."
    #The name is always on the line(s) immediately after "Name" and before "Father's /Guardian"
    # Extract everything between "Name\n" and "Father's"
    match = re.search(
        r"\bName\n(.+?)\nFather",
        page_text,
        re.DOTALL
    )
    if match:
        raw = match.group(1).strip()
        name = " ".join(raw.split())   # collapse multiline names like "SALMAN\nQAMAR"
        return sanitize_filename(name)
    return "Unknown_Candidate"

import fitz   # PyMuPDF gives consistent text layout
SPLIT_DIR  = "/content/drive/MyDrive/TALASH/split_cvs"
os.makedirs(SPLIT_DIR, exist_ok=True)
os.system(f"rm -rf {SPLIT_DIR}/*")

doc = fitz.open(SOURCE_PDF)

# Group pages into CVs
cv_groups = []
current_group = []

for i in range(len(doc)):
    text = doc[i].get_text("text")
    if is_new_cv_start(text):
        if current_group:
            cv_groups.append(current_group)
        current_group = [i]
    else:
        if current_group:
            current_group.append(i)

if current_group:
    cv_groups.append(current_group)

print(f"Total CVs detected: {len(cv_groups)}")

#Save each CV group as one multi-page PDF
saved = 0
skipped = 0

#need pypdf to write the output PDFs
reader = PdfReader(SOURCE_PDF)

for cv_idx, page_indices in enumerate(cv_groups):
    first_page_text = doc[page_indices[0]].get_text("text")
    candidate_name  = extract_name_from_text(first_page_text)

    if candidate_name == "Unknown_Candidate":
        skipped += 1
        print(f"  Skipped group {cv_idx+1} (pages {page_indices}) — no name found")
        continue

    writer = PdfWriter()
    for page_idx in page_indices:
        writer.add_page(reader.pages[page_idx])

    filename = f"CV_{cv_idx+1:03d}_{candidate_name}.pdf"
    out_path = os.path.join(SPLIT_DIR, filename)
    with open(out_path, "wb") as f:
        writer.write(f)

    print(f"  Saved: {filename}  ({len(page_indices)} pages)")
    saved += 1

print(f"\nDone. {saved} CVs saved, {skipped} skipped.")

Total CVs detected: 43
  Saved: CV_001_MUHAMMAD_SALMAN_QAMAR.pdf  (4 pages)
  Saved: CV_002_Aamina_Akbar.pdf  (4 pages)
  Saved: CV_003_MUHAMMAD_SHAHWAZ.pdf  (3 pages)
  Saved: CV_004_shaheer.pdf  (4 pages)
  Saved: CV_005_Nasir_Ali_Shah.pdf  (5 pages)
  Saved: CV_006_Muhammad_Farrukh_Qureshi.pdf  (10 pages)
  Saved: CV_007_Idris_Khan.pdf  (5 pages)
  Saved: CV_008_MUHAMMAD_MAJID_AZIZ.pdf  (4 pages)
  Saved: CV_009_Muhammad_Sarmad_Tariq.pdf  (5 pages)
  Saved: CV_010_Farman_Ali.pdf  (4 pages)
  Saved: CV_011_Fiza_Murtaza.pdf  (7 pages)
  Saved: CV_012_tayyaba_gull_tareen.pdf  (5 pages)
  Saved: CV_013_Muhammad_Abubakar.pdf  (8 pages)
  Saved: CV_014_Fahd_Sikandar_Khan.pdf  (6 pages)
  Saved: CV_015_Usman_Masud.pdf  (18 pages)
  Saved: CV_016_Muhammad_Ehatisham_ul_Haq.pdf  (4 pages)
  Saved: CV_017_Waqas_Amin.pdf  (10 pages)
  Saved: CV_018_SYED_TAMOORULHASSAN.pdf  (5 pages)
  Saved: CV_019_Manzoor_Ellahi.pdf  (9 pages)
  Saved: CV_020_Sarosh_Tahir.pdf  (6 pages)
  Saved: CV_021_M_Usman

In [22]:
from google.colab import userdata
#CONFIG
CV_FOLDER = SPLIT_DIR      #folder with CV PDFs
OUTPUT_DIR  = "/content/drive/MyDrive/TALASH/output"
API_KEY     = userdata.get('LLM_API_KEY')

MODEL_NAME = "gemini-3.1-flash-lite-preview"

os.makedirs(OUTPUT_DIR, exist_ok=True)
client = genai.Client(api_key=API_KEY)

print("Configuration loaded")
print(f"Model: {MODEL_NAME}")
print(f"CV folder: {CV_FOLDER}")
print(f"Output dir: {OUTPUT_DIR}")

Configuration loaded
Model: gemini-3.1-flash-lite-preview
CV folder: /content/drive/MyDrive/TALASH/split_cvs
Output dir: /content/drive/MyDrive/TALASH/output


### SYSTEM PROMPT

In [23]:
SYSTEM_PROMPT = """You are an expert CV parser for a university recruitment system called TALASH
(Talent Acquisition and Learning Automation for Smart Hiring) built as part of a CS417 Large
Language Models course project

Your job is to extract ALL structured information from a CV and return ONLY a valid JSON object
Follow these rules strictly:
- Return ONLY raw JSON. No explanation, no preamble, no markdown, no triple backticks
- If a field is not found in the CV, use null for scalar fields and [] for list fields
- Never invent or hallucinate information not present in the CV
- Be thorough — missing fields will cause downstream analysis modules to fail
- For dates, preserve the original format found in the CV (e.g. "2019", "Jan 2019", "2019-01").
- For author lists, always store as arrays even if only one author"""

### CV Parsing Logic & Prompt Engineering

In [24]:
def build_extraction_prompt(cv_text: str) -> str:
    return f"""Extract all information from the CV below into this exact JSON structure.
Return ONLY the JSON object with no preamble or markdown. Use null for missing values.

{{
  "personal_info": {{
    "full_name": "string",
    "fathers_name": "string",
    "spouse_name": "string",
    "date_of_birth": "string",
    "nationality": "string",
    "email": "string",
    "phone": "string",
    "address": "string",
    "current_salary": "string",
    "expected_salary": "string"
  }},

  "education": [
    {{
      "degree_title": "e.g. BS Computer Science",
      "passing_year": "string",
      "institution": "string",
      "marks_percentage_cgpa": "string",
      "division_grade": "string"
    }}
  ],

  "experience": [
    {{
      "job_title": "string",
      "organization": "string",
      "location": "string",
      "start_date": "string",
      "end_date": "string",
      "duration": "string",
      "responsibilities": []
    }}
  ],

  "skills": {{
    "technical_skills": [],
    "programming_languages": [],
    "tools": []
  }},

  "publications": {{
    "journal_papers": [
      {{
        "title": "string",
        "journal_name": "string",
        "year": "string",
        "authors": []
      }}
    ],
    "conference_papers": [
      {{
        "title": "string",
        "conference_name": "string",
        "year": "string",
        "authors": []
      }}
    ]
  }},

  "supervision": {{
    "ms_students_count": "integer",
    "phd_students_count": "integer",
    "student_details": [
      {{ "name": "string", "degree": "MS|PhD", "year": "string", "status": "Completed|Ongoing" }}
    ]
  }},

  "projects": [
    {{ "title": "string", "funding_agency": "string", "amount": "string", "role": "string" }}
  ],

  "awards_and_certificates": [
    {{ "title": "string", "year": "string", "organization": "string" }}
  ]
}}

CV TEXT:
\"\"\"
{cv_text}
\"\"\"
"""

### PDF Text Extractor

In [25]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extracts all text from a PDF using pypdf page by page.
    Returns the full text as a single string.
    """
    reader = PdfReader(pdf_path)
    pages = []

    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text()
        if text:
            text = text.strip()
            pages.append(f"[Page {i}]\n{text}")

    return "\n\n".join(pages)

### LLM Extraction Function

In [31]:
from google.genai import types

def extract_structured_data(cv_text: str, candidate_name: str = "Unknown") -> dict:
    prompt = build_extraction_prompt(cv_text)

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                response_mime_type='application/json',
                max_output_tokens=8192
            )
        )

        raw_text = response.text.strip()
        raw_text = re.sub(r"^```json\s*|^```\s*|\s*```$", "", raw_text, flags=re.MULTILINE)

        data = json.loads(raw_text)
        print(f"Extraction successful: {candidate_name}")
        return data

    except json.JSONDecodeError as e:
        print(f"JSON parse error for {candidate_name}: {e}")
        debug_path = os.path.join(OUTPUT_DIR, f"DEBUG_{candidate_name}.txt")
        with open(debug_path, "w", encoding="utf-8") as f:
            f.write(raw_text)
        return {}
    except Exception as e:
        print(f"API error for {candidate_name}: {e}")
        return {}

### Flatten Functions (nested JSON → flat CSV rows)

In [32]:
# One function per output table
def flatten_personal_info(data: dict, source_file: str) -> dict:
    if isinstance(data, list):
        data = data[0] if len(data) > 0 else {}
    p = data.get("personal_info") or {}
    return {
        "source_file":     source_file,
        "full_name":       p.get("full_name"),
        "fathers_name":    p.get("fathers_name"),
        "spouse_name":     p.get("spouse_name"),
        "date_of_birth":   p.get("date_of_birth"),
        "nationality":     p.get("nationality"),
        "email":           p.get("email"),
        "phone":           p.get("phone"),
        "address":         p.get("address"),
        "current_salary":  p.get("current_salary"),
        "expected_salary": p.get("expected_salary")
    }


def flatten_education(data: dict, source_file: str) -> list:
    if isinstance(data, list) and len(data) > 0: data = data[0]
    rows = []
    for edu in (data.get("education") or []):
        rows.append({
            "source_file":           source_file,
            "degree_title":          edu.get("degree_title"),
            "passing_year":          edu.get("passing_year"),
            "institution":           edu.get("institution"),
            "marks_percentage_cgpa": edu.get("marks_percentage_cgpa"),
            "division_grade":        edu.get("division_grade")
        })
    return rows


def flatten_experience(data: dict, source_file: str) -> list:
    if isinstance(data, list) and len(data) > 0: data = data[0]
    rows = []
    for exp in (data.get("experience") or []):
        rows.append({
            "source_file":      source_file,
            "job_title":        exp.get("job_title"),
            "organization":     exp.get("organization"),
            "location":         exp.get("location"),
            "start_date":       exp.get("start_date"),
            "end_date":         exp.get("end_date"),
            "duration":         exp.get("duration"),
            "responsibilities": "; ".join(exp.get("responsibilities") or [])
        })
    return rows


def flatten_skills(data: dict, source_file: str) -> dict:
    if isinstance(data, list) and len(data) > 0: data = data[0]
    s = data.get("skills") or {}
    return {
        "source_file":           source_file,
        "technical_skills":      "; ".join(s.get("technical_skills") or []),
        "programming_languages": "; ".join(s.get("programming_languages") or []),
        "tools":                 "; ".join(s.get("tools") or [])
    }


def flatten_journal_papers(data: dict, source_file: str) -> list:
    if isinstance(data, list) and len(data) > 0: data = data[0]
    rows = []
    pubs = data.get("publications") or {}
    for paper in (pubs.get("journal_papers") or []):
        rows.append({
            "source_file":  source_file,
            "title":        paper.get("title"),
            "journal_name": paper.get("journal_name"),
            "year":         paper.get("year"),
            "authors":      "; ".join(paper.get("authors") or [])
        })
    return rows


def flatten_conference_papers(data: dict, source_file: str) -> list:
    if isinstance(data, list) and len(data) > 0: data = data[0]
    rows = []
    pubs = data.get("publications") or {}
    for paper in (pubs.get("conference_papers") or []):
        rows.append({
            "source_file":     source_file,
            "title":           paper.get("title"),
            "conference_name": paper.get("conference_name"),
            "year":            paper.get("year"),
            "authors":         "; ".join(paper.get("authors") or [])
        })
    return rows


def flatten_supervision(data: dict, source_file: str) -> dict:
    if isinstance(data, list) and len(data) > 0: data = data[0]
    s = data.get("supervision") or {}
    students = s.get("student_details") or []
    students_str = "; ".join(
        f"{st.get('name','?')} ({st.get('degree','?')}, {st.get('status','?')})"
        for st in students
    )
    return {
        "source_file":         source_file,
        "ms_students_count":   s.get("ms_students_count"),
        "phd_students_count":  s.get("phd_students_count"),
        "supervised_students": students_str,
    }

def flatten_projects(data: dict, source_file: str) -> list:
    if isinstance(data, list) and len(data) > 0: data = data[0]
    rows = []
    for proj in (data.get("projects") or []):
        rows.append({
            "source_file":    source_file,
            "title":          proj.get("title"),
            "role":           proj.get("role"),
            "funding_agency": proj.get("funding_agency"),
            "amount":         proj.get("amount") # Simplified key
        })
    return rows

def flatten_awards_and_certs(data: dict, source_file: str) -> list:
    """Combined awards and certificates as per the Revised Schema"""
    if isinstance(data, list) and len(data) > 0: data = data[0]
    rows = []
    for item in (data.get("awards_and_certificates") or []):
        rows.append({
            "source_file":  source_file,
            "title":        item.get("title"),
            "organization": item.get("organization"),
            "year":         item.get("year")
        })
    return rows

### Missing Information Detector + Email Drafter

In [33]:
def detect_missing_fields(data: dict) -> list:
    """
    Checks critical fields needed by TALASH.
    Optimized to match the Revised Extraction Schema.
    """
    missing = []

    # 1. Personal Info
    pi = data.get("personal_info") or {}
    if not pi.get("full_name"): missing.append("Full name")
    if not pi.get("email"): missing.append("Email address")
    if not pi.get("phone"): missing.append("Phone number")

    # 2. Education (Updated keys to match Cell 5/8)
    edu = data.get("education") or []
    if not edu:
        missing.append("Education records (degrees and institutions)")
    else:
        for e in edu:
            if not e.get("marks_percentage_cgpa"):
                level = e.get("degree_title") or "degree"
                missing.append(f"Marks/CGPA for {level}")
            if not e.get("passing_year"):
                level = e.get("degree_title") or "degree"
                missing.append(f"Passing year for {level}")

    # 3. Experience
    exp = data.get("experience") or []
    if not exp:
        missing.append("Professional experience / employment history")
    else:
        for e in exp:
            if not e.get("start_date") or not e.get("end_date"):
                role = e.get("job_title") or "position"
                missing.append(f"Dates of employment for {role}")

    # 4. Publications (Simplified - No DOI/ISSN check)
    pubs = data.get("publications") or {}
    if not pubs.get("journal_papers") and not pubs.get("conference_papers"):
        missing.append("Research publications (Journal or Conference papers)")

    return missing


def generate_missing_info_email(candidate_name: str, missing_fields: list) -> str:
    """
    Drafts a professional personalized email requesting missing CV info.
    Required by Section 4 (Functional Requirement 4) of the project PDF.
    """
    if not missing_fields:
        return ""

    name = candidate_name or "Applicant"
    fields_list = "\n".join(f"    • {field}" for field in missing_fields)

    return f"""Subject: Additional Information Required — Your Application (TALASH Recruitment System)

Dear {name},

Thank you for submitting your application and CV for consideration through the TALASH
Smart HR Recruitment System. We appreciate your interest and the time you have taken
to prepare your profile.

Upon automated review of your submitted CV, our system has identified that the following
information is either missing or incomplete:

{fields_list}

This information is required to complete a thorough and fair evaluation of your academic,
research, and professional profile. Without it, certain aspects of your application cannot
be fully assessed, which may affect your overall evaluation.

We kindly request that you:
  1. Review the items listed above.
  2. Prepare the relevant supporting details or documents.
  3. Submit an updated CV or reply to this email with the missing information at your
     earliest convenience.

Please note that complete applications are evaluated more thoroughly and may be prioritized
in the review process.

If you have any questions, please do not hesitate to contact us.

Best regards,
TALASH Recruitment Team
Faculty of Computing, SEECS""".strip()


### MAIN PIPELINE

In [34]:
def process_all_cvs(cv_folder: str, output_dir: str):
    p = Path(cv_folder)
    pdf_files = sorted(list(p.glob("*.pdf")))

    # Accumulators for REQUIRED tables only
    all_personal, all_education, all_experience = [], [], []
    all_skills, all_journals, all_conferences = [], [], []
    all_supervision, all_projects, all_awards = [], [], []
    all_missing = []

    for pdf_path in tqdm(pdf_files, desc="Extracting Data"):
        filename = pdf_path.name

        cv_text = extract_text_from_pdf(str(pdf_path))

        structured = extract_structured_data(cv_text, candidate_name=filename)
        if not structured:
            continue

        if isinstance(structured, list) and len(structured) > 0:
            structured = structured[0]

        pi = structured.get("personal_info") or {}
        raw_name = pi.get("full_name") or filename
        clean_name = re.sub(r"[^\w\s-]", "", raw_name).strip().replace(" ", "_")

        json_path = os.path.join(output_dir, f"{clean_name}.json")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(structured, f, indent=2)

        try:
            all_personal.append(flatten_personal_info(structured, clean_name))
            all_education.extend(flatten_education(structured, clean_name))
            all_experience.extend(flatten_experience(structured, clean_name))
            all_skills.append(flatten_skills(structured, clean_name))
            all_journals.extend(flatten_journal_papers(structured, clean_name))
            all_conferences.extend(flatten_conference_papers(structured, clean_name))
            all_supervision.append(flatten_supervision(structured, clean_name))
            all_projects.extend(flatten_projects(structured, clean_name))
            all_awards.extend(flatten_awards_and_certs(structured, clean_name))
        except Exception as e:
            print(f" Flattening error for {clean_name}: {e}")

        missing = detect_missing_fields(structured)
        if missing:
            all_missing.append({
                "candidate": raw_name,
                "source_file": filename,
                "missing_fields": "; ".join(missing),
                "draft_email": generate_missing_info_email(raw_name, missing),
            })

        time.sleep(1)

    tables = {
        "01_personal_info": all_personal,
        "02_education": all_education,
        "03_experience": all_experience,
        "04_skills": all_skills,
        "05_journal_papers": all_journals,
        "06_conf_papers": all_conferences,
        "07_supervision": all_supervision,
        "08_projects": all_projects,
        "09_awards_and_certs": all_awards,
    }

    if not any(tables.values()):
        print("No data extracted.")
        return

    for table_name, rows in tables.items():
        if rows:
            pd.DataFrame(rows).to_csv(os.path.join(output_dir, f"{table_name}.csv"), index=False)

    excel_path = os.path.join(output_dir, "TALASH_Preprocessed_Data.xlsx")
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        for table_name, rows in tables.items():
            if rows:
                pd.DataFrame(rows).to_excel(writer, sheet_name=table_name[3:31], index=False)

    if all_missing:
        pd.DataFrame(all_missing).to_csv(os.path.join(output_dir, "10_missing_info.csv"), index=False)
        print(f"\nPipeline complete! Outputs saved to {output_dir}")

In [35]:
#Run the Pipeline
process_all_cvs(CV_FOLDER, OUTPUT_DIR)

Extracting Data:   0%|          | 0/43 [00:00<?, ?it/s]

Extraction successful: CV_001_MUHAMMAD_SALMAN_QAMAR.pdf


Extracting Data:   2%|▏         | 1/43 [00:11<08:09, 11.66s/it]

Extraction successful: CV_002_Aamina_Akbar.pdf


Extracting Data:   5%|▍         | 2/43 [00:17<05:43,  8.37s/it]

...

API error for CV_041_Ammara_Nasim.pdf: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Extraction successful: CV_042_Ahmed_Naeem.pdf


Extracting Data:  98%|█████████▊| 42/43 [06:32<00:10, 10.41s/it]

Extraction successful: CV_043_MUHAMMAD_ADEEL_ALTAF.pdf


Extracting Data: 100%|██████████| 43/43 [06:39<00:00,  9.30s/it]



Pipeline complete! Outputs saved to /content/drive/MyDrive/TALASH/output
